In [1]:
import pandas as pd

df = pd.read_parquet(
    r"C:\Users\nukal\Downloads\AI_Email_Deliverability_Intelligence\content\drive\MyDrive\AI_Email_Deliverability_Intelligence\data\processed\sendguard_base_processed.parquet"
)

print(df.shape)

(908226, 134)


In [2]:
def create_status(rate):
    if rate >= 98:
        return "Excellent"
    elif rate >= 95:
        return "Good"
    elif rate >= 90:
        return "Warning"
    else:
        return "Critical"

df["deliverability_status"] = df["delivery_rate"].apply(create_status)

In [3]:
# Base columns to remove
drop_cols = [
    "deliverability_status",
    "delivery_rate",
    "client_id",
    "campaign_id",
    "campaign_sent_time",
    "open_rate",
    "click_rate",
    "soft_bounce_rate",
    "hard_bounce_rate",
    "complaint_rate",
    "phishing_rate"
]

# Remove ONLY current campaign outcome counts
current_outcome_cols = [
    "campaign_opens_count",
    "campaign_unique_opens_count",
    "campaign_clicks_count",
    "campaign_unique_clicks_count",
    "campaign_soft_bounced_count",
    "campaign_hard_bounced_count",
    "campaign_complaint_count",
    "campaign_phishing_count"
]

drop_cols.extend(current_outcome_cols)

X = df.drop(columns=drop_cols)
y = df["deliverability_status"]

print("Remaining Features:", X.shape)

Remaining Features: (908226, 116)


In [4]:
import numpy as np

# Convert object columns to category
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("category")

# Reduce integer memory
for col in df.select_dtypes(include="int64").columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")

# Reduce float memory
for col in df.select_dtypes(include="float64").columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

342.34465980529785 MB


In [34]:
print(X.dtypes["client_tag_num"])

int8


In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=drop_cols)
y = df["deliverability_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["int64", "float64", "bool"]
).columns.tolist()

print("Categorical:", len(categorical_cols))
print("Numerical :", len(numerical_cols))
print("\nCategorical Columns:")
print(categorical_cols)

Categorical: 0
Numerical : 2

Categorical Columns:
[]


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

# Detect all categorical columns correctly
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Build preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

# Fit on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Categorical columns:", categorical_cols)
print("Total categorical columns:", len(categorical_cols))
print("Processed shape:", X_train_processed.shape)

Categorical columns: ['client_tag_name', 'campaign_test_part', 'campaign_test_type', 'domain_in_links_min_creation_time', 'domain_in_links_max_creation_time', 'domain_in_links_avg_creation_time', 'domain_in_links_most_used_creation_time', 'sent_date', 'sent_day_name', 'has_valid_audience', 'zero_audience_with_activity']
Total categorical columns: 11
Processed shape: (726580, 116)


In [29]:
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

classes = np.unique(y_train_encoded)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_encoded
)

class_weight_dict = dict(zip(classes, weights))

print(label_encoder.classes_)
print(class_weight_dict)

['Critical' 'Excellent' 'Good' 'Warning']
{np.int64(0): np.float64(3.3296366902518604), np.int64(1): np.float64(0.3082616044639342), np.int64(2): np.float64(3.3793161184700105), np.int64(3): np.float64(6.259735336687573)}


In [10]:
import xgboost
print(xgboost.__version__)

3.3.0


In [30]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=250,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist",
    eval_metric="mlogloss"
)

sample_weights = np.array([class_weight_dict[i] for i in y_train_encoded])

model.fit(
    X_train_processed,
    y_train_encoded,
    sample_weight=sample_weights
)

print("V3 Model trained successfully!")

V3 Model trained successfully!


In [31]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_processed)

accuracy = accuracy_score(y_test_encoded, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n")

print(
    classification_report(
        y_test_encoded,
        y_pred,
        target_names=label_encoder.classes_
    )
)

Accuracy: 0.9927

Classification Report:

              precision    recall  f1-score   support

    Critical       0.99      0.99      0.99     13639
   Excellent       1.00      0.99      1.00    147315
        Good       0.94      0.99      0.96     13438
     Warning       0.95      0.98      0.96      7254

    accuracy                           0.99    181646
   macro avg       0.97      0.99      0.98    181646
weighted avg       0.99      0.99      0.99    181646



In [32]:
# Create dtype map from the original feature dataframe
dtype_map = X.dtypes.astype(str).to_dict()

print("dtype_map created successfully!")

dtype_map created successfully!


In [33]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/xgboost_model_v3.pkl")
joblib.dump(preprocessor, "models/preprocessor_v3.pkl")
joblib.dump(label_encoder, "models/label_encoder_v3.pkl")

schema = {
    "feature_names": X.columns.tolist(),
    "dtypes": X.dtypes.astype(str).to_dict(),
    "target_classes": label_encoder.classes_.tolist()
}

joblib.dump(schema, "models/schema_v3.pkl")

print("✅ All V3 artifacts saved successfully!")

✅ All V3 artifacts saved successfully!
